In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd
import json
import os

class VectorDB:
    """
    Vector Database System - stores embeddings and performs similarity search
    Similar to textToVector.py pattern but without ollama
    """
    def __init__(self, vector_db_path: str = 'vector_db.json'):
        self.vector_db_path = vector_db_path
        self.vectors = []
        self.vectorizer = None
        self.tfidf_matrix = None
    
    def add_text(self, texts: list):
        """Add texts to vector database"""
        self.vectors = texts
        self.vectorizer = TfidfVectorizer(
            lowercase=True, 
            stop_words='english', 
            ngram_range=(1, 2),
            max_features=500
        )
        self.tfidf_matrix = self.vectorizer.fit_transform(texts)
    
    def retrieve(self, query: str, top_n: int = 5) -> list:
        """Retrieve most similar items from database - like api.py retrieve()"""
        if self.vectorizer is None:
            return []
        
        query_vector = self.vectorizer.transform([query])
        similarities = cosine_similarity(query_vector, self.tfidf_matrix)[0]
        
        # Get top N results
        top_indices = np.argsort(similarities)[::-1][:top_n]
        results = []
        
        for idx in top_indices:
            if similarities[idx] > 0:
                results.append({
                    'text': self.vectors[idx],
                    'similarity': float(similarities[idx]),
                    'index': int(idx)
                })
        
        return results
    
    def save(self):
        """Save vector database to file"""
        data = {
            'vectors': self.vectors,
            'metadata': {'count': len(self.vectors)}
        }
        os.makedirs(os.path.dirname(self.vector_db_path) or '.', exist_ok=True)
        with open(self.vector_db_path, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
    
    def load(self):
        """Load vector database from file"""
        if os.path.exists(self.vector_db_path):
            with open(self.vector_db_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
                self.vectors = data['vectors']
                self.add_text(self.vectors)

class DatasetQA:
    """
    Professional Q&A System using RAG (Retrieval-Augmented Generation) pattern
    Similar to api.py + textToVector.py architecture but without ollama
    """
    
    def __init__(self, df: pd.DataFrame, vector_db_path: str = 'vector_db.json'):
        self.df = df
        self.vector_db_path = vector_db_path
        
        # Initialize metadata
        self.metadata = self._extract_metadata()
        
        # Initialize vector database for retrieval
        self.vector_db = VectorDB(vector_db_path)
        self._initialize_knowledge_base()
    
    def detect_column_types(self, df: pd.DataFrame): 
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
        datetime_cols = []
        
        # Simple heuristic for potential datetime strings
        date_regex = r'(\d{4}[-/]\d{2}[-/]\d{2})|(\d{2}[-/]\d{2}[-/]\d{4})'
        for col in categorical_cols:
            sample = df[col].dropna().head(10).astype(str)
            if sample.str.contains(date_regex).any():
                try:
                    pd.to_datetime(sample, errors='raise')
                    datetime_cols.append(col)
                except:
                    pass
        
        # Move detected datetimes from categorical
        categorical_cols = [c for c in categorical_cols if c not in datetime_cols]
        return numeric_cols, categorical_cols, datetime_cols
    
    def _extract_metadata(self) -> dict:
        """Extract and cache dataset metadata for efficient querying"""
        numeric_cols, categorical_cols, datetime_cols = self.detect_column_types(self.df)
        
        return {
            'shape': self.df.shape,
            'total_rows': len(self.df),
            'total_columns': len(self.df.columns),
            'columns': self.df.columns.tolist(),
            'numeric_columns': numeric_cols,
            'categorical_columns': categorical_cols,
            'datetime_columns': datetime_cols,
            'dtypes': dict(self.df.dtypes),
            'missing_values': dict(self.df.isnull().sum()),
            'duplicates': self.df.duplicated().sum(),
            'memory_usage_mb': self.df.memory_usage(deep=True).sum() / (1024**2),
            'correlation_matrix': self.df[numeric_cols].corr() if len(numeric_cols) >= 2 else None
        }
    
    def _initialize_knowledge_base(self):
        """Initialize knowledge base with FAQ patterns - like textToVector.py prep step"""
        self.knowledge_base = [
            # Dataset size questions
            "how many rows records instances samples are in the dataset",
            "total number of records in this dataset",
            "what is the row count of the data",
            
            # Column/feature questions
            "how many columns features are there",
            "total number of columns attributes fields",
            "what are the column names feature names",
            "list all columns in the dataset",
            
            # Data type questions
            "what are the numeric columns number columns",
            "which columns are numerical features",
            "what are the categorical columns text columns",
            "show me column data types dtypes",
            
            # Data quality
            "are there missing values null values in the dataset",
            "how many missing values incomplete data",
            "check for duplicate records repeated rows",
            "how many duplicates are in the dataset",
            
            # Statistics
            "show me statistics summary of the data",
            "describe the dataset numerical statistics",
            "what are the correlations between features",
            "show top correlations in the dataset",
            
            # Memory/Size
            "what is the memory usage dataset size",
            "how large is the dataset disk space",
            "what is the total number of cells",
            
            # General info
            "give me an overview of the dataset",
            "tell me about the data structure",
            "analyze the dataset quality"
        ]
        
        # Initialize intent system with semantic signatures
        self.intent_handlers = {
            'row_count': {
                'patterns': ['row', 'record', 'instance', 'sample', 'how many rows', 'total records', 'row count'],
                'handler': self._answer_row_count
            },
            'column_count': {
                'patterns': ['column', 'feature', 'field', 'attribute', 'how many columns', 'total features'],
                'handler': self._answer_column_count
            },
            'column_list': {
                'patterns': ['column name', 'feature name', 'list column', 'show column', 'all column'],
                'handler': self._answer_list_columns
            },
            'numeric_columns': {
                'patterns': ['numeric column', 'number column', 'numerical feature', 'numeric feature'],
                'handler': self._answer_numeric_columns
            },
            'categorical_columns': {
                'patterns': ['categorical column', 'category column', 'text column', 'object column'],
                'handler': self._answer_categorical_columns
            },
            'missing_data': {
                'patterns': ['missing value', 'null value', 'missing data', 'nan', 'incomplete'],
                'handler': self._answer_missing_values
            },
            'duplicates': {
                'patterns': ['duplicate', 'repeated', 'duplicate record', 'duplicated row'],
                'handler': self._answer_duplicates
            },
            'statistics': {
                'patterns': ['statistic', 'summary', 'describe', 'mean', 'std dev', 'quantile'],
                'handler': self._answer_statistics
            },
            'data_types': {
                'patterns': ['data type', 'dtype', 'column type', 'type of column'],
                'handler': self._answer_data_types
            },
            'correlation': {
                'patterns': ['correlation', 'correlate', 'relationship', 'correlated', 'relate'],
                'handler': self._answer_correlation
            },
            'memory': {
                'patterns': ['memory', 'size', 'cell', 'disk', 'large', 'storage'],
                'handler': self._answer_memory_usage
            }
        }
        
        # Prepare context vectorizer for semantic matching
        all_patterns = []
        for intent_name, intent_data in self.intent_handlers.items():
            all_patterns.extend(intent_data['patterns'])
        
        self.context_vectorizer = TfidfVectorizer(
            lowercase=True,
            stop_words='english',
            ngram_range=(1, 2),
            max_features=300
        )
        self.context_vectorizer.fit(all_patterns)
        
        # Add text to vector database for retrieval
        self.vector_db.add_text(self.knowledge_base)
    
    def _classify_handler_by_context(self, context_text: str) -> callable:
        """
        Semantically classify which handler to use based on retrieved context
        Uses TF-IDF + cosine similarity - learns to map context to handlers
        """
        # Vectorize the context
        context_vector = self.context_vectorizer.transform([context_text.lower()])
        
        handler_scores = {}
        
        # Calculate similarity between context and each handler's patterns
        for intent_name, intent_data in self.intent_handlers.items():
            patterns_vectors = self.context_vectorizer.transform(intent_data['patterns'])
            similarities = cosine_similarity(context_vector, patterns_vectors)[0]
            max_similarity = np.max(similarities) if len(similarities) > 0 else 0
            handler_scores[intent_name] = max_similarity
        
        # Return handler with highest semantic similarity
        best_handler_name = max(handler_scores, key=handler_scores.get)
        best_score = handler_scores[best_handler_name]
        
        # If confidence is high enough, return the handler
        if best_score > 0.2:
            return self.intent_handlers[best_handler_name]['handler']
        
        # Fallback to list columns
        return self._answer_list_columns
    
    def answer(self, question: str) -> str:
        """
        Answer user questions using RAG + Semantic Handler Classification
        Learns to route to appropriate handler based on context similarity
        """
        # Check for column-specific questions first (highest priority)
        for col in self.metadata['columns']:
            if col.lower() in question.lower():
                return self._answer_column_specific(col)
        
        # Retrieve most similar questions from knowledge base
        retrieved = self.vector_db.retrieve(question, top_n=3)
        
        if not retrieved:
            return self._answer_unknown_question()
        
        # Get the most relevant context (highest similarity)
        best_match = retrieved[0]
        best_text = best_match['text'].lower()
        
        # Semantically classify which handler to use based on context
        # This learns to map context patterns to handlers without hardcoded if-else
        handler = self._classify_handler_by_context(best_text)
        
        # Call the selected handler
        return handler()

In [ ]:
# ============================================================================
# RAG-STYLE Q&A SYSTEM - Dataset Analysis
# Similar to api.py + textToVector.py architecture
# ============================================================================
df_clean = pd.read_csv('llm_rag_dataset_6k.csv.csv')  # Load your cleaned dataset here
# Initialize the Q&A system with RAG (Retrieval-Augmented Generation)
qa_system = DatasetQA(df_clean, vector_db_path='vector_db/vector_db.json')

# Save vector database for future use (like textToVector.py)
qa_system.vector_db.save()
print("✓ Vector database saved to 'vector_db/vector_db.json'")

# Example queries demonstrating RAG-style retrieval and generation
print("\n" + "="*80)
print("DATASET Q&A SYSTEM - RAG Pattern (Retrieval-Augmented Generation)")
print("="*80)

# Query 1: Dataset Structure
print("\n[Q] How many rows and columns are in this dataset?")
answer1 = qa_system.answer("How many rows do we have?")
print("[A]", answer1)
print("\n" + "-"*80)

# Query 2: Column Information
print("\n[Q] What are the column names?")
answer2 = qa_system.answer("What columns are there?")
print("[A]", answer2)
print("\n" + "-"*80)

# Query 3: Data Types
print("\n[Q] What are the numeric features?")
answer3 = qa_system.answer("Which columns are numeric?")
print("[A]", answer3)
print("\n" + "-"*80)

# Query 4: Data Quality
print("\n[Q] Are there any missing values in the dataset?")
answer4 = qa_system.answer("Tell me about missing values")
print("[A]", answer4)
print("\n" + "-"*80)

# Query 5: Duplicate Check
print("\n[Q] Are there duplicate records?")
answer5 = qa_system.answer("How many duplicates?")
print("[A]", answer5)
print("\n" + "-"*80)

# Query 6: Statistics
print("\n[Q] Provide statistical summary of the data")
answer6 = qa_system.answer("Show me statistics")
print("[A]", answer6)
print("\n" + "-"*80)

# Query 7: Correlations
print("\n[Q] What are the correlations between numeric features?")
answer7 = qa_system.answer("Correlation analysis")
print("[A]", answer7)
print("\n" + "-"*80)


/tmp/ipykernel_99074/3478672274.py:89: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
/tmp/ipykernel_99074/3478672274.py:96: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  if sample.str.contains(date_regex).any():
/tmp/ipykernel_99074/3478672274.py:96: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  if sample.str.conta

AttributeError: 'DatasetQA' object has no attribute '_answer_row_count'